Notebook 1 — EDA & Cleaning

Cell 1 — Setup & Load

Import pandas, numpy, matplotlib, seaborn
Load the CSV
Print shape, dtypes, describe()

In [40]:
import pandas as pd 
import numpy 
import matplotlib 
import seaborn 

df = pd.read_csv('../data/climate_change_impact_on_agriculture_2024.csv')
print(f"The shape of the data: {df.shape}")
print(f"\nThe data types of the columns of the table: \n{df.dtypes}")
print(f"\nFirst 5 rows of the table: \n{df.head()}")

The shape of the data: (10000, 15)

The data types of the columns of the table: 
Year                             int64
Country                         object
Region                          object
Crop_Type                       object
Average_Temperature_C          float64
Total_Precipitation_mm         float64
CO2_Emissions_MT               float64
Crop_Yield_MT_per_HA           float64
Extreme_Weather_Events           int64
Irrigation_Access_%            float64
Pesticide_Use_KG_per_HA        float64
Fertilizer_Use_KG_per_HA       float64
Soil_Health_Index              float64
Adaptation_Strategies           object
Economic_Impact_Million_USD    float64
dtype: object

First 5 rows of the table: 
   Year Country         Region  Crop_Type  Average_Temperature_C  \
0  2001   India    West Bengal       Corn                   1.55   
1  2024   China          North       Corn                   3.23   
2  2001  France  Ile-de-France      Wheat                  21.11   
3  2001  Canada    

Cell 2 — Missing values

Check NaN count per column
Check percentage missing
Handle any missing values appropriately

In [17]:
nan_counts = df.isnull().sum()
print(f"The existing NaN values: \n{nan_counts}")

total_cells = df.size
print(f"\nThe number of total cells: {total_cells}")

total_missing = df.isnull().sum().sum()
print(f"The number of total missing cells: {total_missing}")

total_pct = (total_missing / total_cells) * 100
print(f"The total percentage of missing cells: {total_pct}%")

if df.isnull().values.any():
    df.fillna("Unknown", inplace=True)
    print("Missing values are filled with 'Unknown")
else:
    print('No null values to fill or replace')

The existing NaN values: 
Year                           0
Country                        0
Region                         0
Crop_Type                      0
Average_Temperature_C          0
Total_Precipitation_mm         0
CO2_Emissions_MT               0
Crop_Yield_MT_per_HA           0
Extreme_Weather_Events         0
Irrigation_Access_%            0
Pesticide_Use_KG_per_HA        0
Fertilizer_Use_KG_per_HA       0
Soil_Health_Index              0
Adaptation_Strategies          0
Economic_Impact_Million_USD    0
dtype: int64

The number of total cells: 150000
The number of total missing cells: 0
The total percentage of missing cells: 0.0%
No null values to fill or replace


Cell 3 — Explore key columns

How many unique countries?
How many unique crop types?
What years does the data cover?
How many unique regions?

In [42]:
unique_countries = sorted(df['Country'].unique())
print(f"Unique countries names: {unique_countries}")
unique_countries_count = df['Country'].nunique()
print(f"Number of countries: {unique_countries_count}")

unique_crops = sorted(df['Crop_Type'].unique())
print(f"\nUnique crop type: {unique_crops}")
unique_crops_count = df["Crop_Type"].nunique()
print(f"Unique crops number: {unique_crops_count}")

unique_year = df['Year'].unique()
print(f"\nYears that data covers: {unique_year}")

unique_regions = df['Region'].unique()
print(f"\nUnique regions: {unique_regions}")
unique_regions_count = df['Region'].nunique()
print(f"Unique regions number: {unique_regions_count}")

Unique countries names: ['Argentina', 'Australia', 'Brazil', 'Canada', 'China', 'France', 'India', 'Nigeria', 'Russia', 'USA']
Number of countries: 10

Unique crop type: ['Barley', 'Coffee', 'Corn', 'Cotton', 'Fruits', 'Rice', 'Soybeans', 'Sugarcane', 'Vegetables', 'Wheat']
Unique crops number: 10

Years that data covers: [2001 2024 1998 2019 1997 2021 2012 2018 2006 1993 2003 1999 1990 2017
 2015 2000 2016 1996 2010 2002 2011 1995 2004 2008 2005 2020 1994 1991
 2022 2007 1992 2013 2023 2014 2009]

Unique regions: ['West Bengal' 'North' 'Ile-de-France' 'Prairies' 'Tamil Nadu' 'Midwest'
 'Northeast' 'New South Wales' 'Punjab' 'North West' 'South East'
 'Grand Est' 'Northwestern' 'Siberian' 'Northwest' 'Victoria'
 'Nouvelle-Aquitaine' 'South' 'Quebec' 'Southeast' 'Ontario' 'East'
 'Pampas' 'Western Australia' 'Volga' 'Maharashtra'
 'Provence-Alpes-Cote d’Azur' 'West' 'Central' 'North Central' 'Patagonia'
 'Queensland' 'South West' 'British Columbia']
Unique regions number: 34


Cell 4 — Feature engineering
Adding new columns:

In [33]:
# Decade column — for trend analysis
df["Decade"] = (df["Year"] // 10) * 10 

# Yield category
df["Yield_Category"] = pd.cut(
    df["Crop_Yield_MT_per_HA"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

# High stress flag — extreme weather AND low soil health
df["High_Stress"] = (
    (df["Extreme_Weather_Events"] >= 7) &
    (df["Soil_Health_Index"] < 50)
)

Cell 5 — Crop types relevant to Central Asia

In [41]:
ca_crops = ["Wheat", "Cotton", "Rice", "Barley"]

ca_df = df[df["Crop_Type"].isin(ca_crops)].copy()
print(f"CA-relevant crops rows: {len(ca_df)}")


CA-relevant crops rows: 4065


Cell 6 — Save clean version to PostgreSQL

In [39]:
# Use .env for password — no hardcoding
import os
from dotenv import load_dotenv
load_dotenv()
from sqlalchemy import create_engine


engine = create_engine(os.getenv("DATABASE_URL"))
df.to_sql("climate_crops", engine, if_exists="replace", index=False)
print(f"✅ {len(df)} rows written to PostgreSQL")

✅ 10000 rows written to PostgreSQL
